# MNIST Vision Transformer on 2× T4 (CUDA + MPI + NCCL + Adam)

This notebook compiles and runs the hand-written CUDA ViT from `train_vit.cu`
on a Kaggle session with **two Tesla T4 GPUs**.

Optimizer: **Adam** (β₁=0.9, β₂=0.999, ε=1e-8, lr=1e-3).
Each run saves a `training_log.csv` with columns `step, elapsed_s, loss, accuracy, t_h2d_ms, t_fwd_ms, t_bwd_ms, t_nccl_ms, t_adam_ms`.

**Setup checklist (do this in the Kaggle UI before running):**

1. **Accelerator → GPU T4 x2** (Settings panel on the right).
2. **Internet → On** (needed once, to `apt-get install` OpenMPI).
3. Optionally add the "Digit Recognizer" dataset; otherwise MNIST is downloaded via torchvision.

Run cells top-to-bottom.
Run cells **1–7** top-to-bottom for analysis.
Section **8** (always last): 8a → 8b for 1-GPU, 8c → 8d for 2-GPU, then 8e to compare curves.


In [ ]:
!rm -rf /tmp/hpc && git clone https://github.com/SadreevAmir/hpc_final_project /tmp/hpc && cp -r /tmp/hpc/. .


## 1. Verify the environment

Expect two T4 GPUs.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## 2. Install OpenMPI

Kaggle ships `libnccl2`, `nvcc`, and `cuBLAS` already. Only `openmpi-bin`/`libopenmpi-dev` are missing.

⚠️ **Do NOT install `libnccl-dev`** — it conflicts with the version bundled in Kaggle's PyTorch image.

In [ ]:
import subprocess
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq',
                'openmpi-bin', 'libopenmpi-dev'], check=True)
print('OK')


## 3. Locate the existing NCCL header and library

NCCL lives inside the PyTorch conda env on Kaggle. We find `nccl.h` and `libnccl.so` and store their dirs in env vars.

In [ ]:
import os, subprocess

hdr = subprocess.check_output(
    'find /opt/conda /usr/include /usr/local -name nccl.h 2>/dev/null | head -n1',
    shell=True, text=True).strip()
lib = subprocess.check_output(
    'find /opt/conda /usr/lib /usr/local -name "libnccl.so*" 2>/dev/null | head -n1',
    shell=True, text=True).strip()

assert hdr, 'nccl.h not found'
assert lib, 'libnccl.so not found'

os.environ['NCCL_INCLUDE'] = os.path.dirname(hdr)
os.environ['NCCL_LIB']     = os.path.dirname(lib)
print('nccl.h     :', hdr)
print('libnccl.so :', lib)


### Toolchain sanity check

In [ ]:
!which nvcc mpicxx mpirun
!nvcc --version | tail -n 2
!mpirun --version | head -n 1
!echo "NCCL include: $NCCL_INCLUDE"
!echo "NCCL lib    : $NCCL_LIB"


## 4. Compile

- `-ccbin mpicxx` — MPI C++ wrapper as host compiler.
- `-arch=sm_75` — Turing (T4).
- `-I$NCCL_INCLUDE` / `-L$NCCL_LIB` / `-rpath` — link against the NCCL we located above.

In [ ]:
!mkdir -p bin
!nvcc -O2 -std=c++17 -ccbin mpicxx -arch=sm_75 \
      -I$NCCL_INCLUDE -L$NCCL_LIB \
      -Xlinker -rpath=$NCCL_LIB \
      src/train_vit.cu -o bin/train_vit \
      -lcublas -lnccl
!ls -lh bin/train_vit
!ldd bin/train_vit | grep -E 'nccl|cublas|mpi'


## 5. Locate (or materialise) the data

Search order: Kaggle digit-recognizer dataset → any `*train*.csv` with 785 columns → torchvision MNIST fallback.

In [ ]:
import os, glob, numpy as np

known = [
    '/kaggle/input/digit-recognizer/train.csv',
    '/kaggle/input/fashionmnist/fashion-mnist_train.csv',
    '/kaggle/input/fashion-mnist/fashion-mnist_train.csv',
]
globbed = sorted(set(
    glob.glob('/kaggle/input/**/*train*.csv', recursive=True) +
    glob.glob('/kaggle/input/**/*Train*.csv', recursive=True)))

def valid_csv(path):
    try:
        with open(path) as f:
            return len(f.readline().split(',')) == 785
    except Exception:
        return False

CSV = next((c for c in known + globbed if os.path.exists(c) and valid_csv(c)), None)

if CSV is None:
    print('Falling back to torchvision MNIST...')
    from torchvision.datasets import MNIST
    ds = MNIST(root='/kaggle/working/mnist_raw', train=True, download=True)
    labels = ds.targets.numpy().astype(np.int32)
    pixels = ds.data.numpy().reshape(-1, 784).astype(np.int32)
    arr    = np.concatenate([labels[:, None], pixels], axis=1)
    header = 'label,' + ','.join(f'pixel{i}' for i in range(784))
    CSV    = '/kaggle/working/train.csv'
    np.savetxt(CSV, arr, fmt='%d', delimiter=',', header=header, comments='')

assert os.path.exists(CSV), CSV
print('Using CSV:', CSV)
print('Size     :', os.path.getsize(CSV) // (1024*1024), 'MiB')
os.environ['CSV'] = CSV
!head -c 120 "$CSV" ; echo
!wc -l "$CSV"


## 6. GPU / memory / NCCL profiling — batch-size sweep

Runs **50 training steps** at batch sizes B ∈ {8, 16, 32, 64} on 1 GPU, plus one run with 2 GPUs at B=32 (if a second GPU is present).

A background thread samples `nvidia-smi` and `psutil` every 250 ms, recording:

- **GPU compute utilization** (%)
- **GPU memory used** (MiB → GiB in plots)
- **GPU power draw** (W)
- **Host CPU utilization** (%)

Training stdout is captured so the cell stays clean. Results are stored in the `prof` dict for the plotting cell below.

⏱ Expected runtime: ~2–4 min total (50 steps × 4 batch sizes × ~10–30 s each).

In [ ]:
import subprocess, threading, time, os
import numpy as np

try:
    import psutil
except ImportError:
    subprocess.run(['pip', 'install', '-q', 'psutil'], check=True)
    import psutil

def _gpu_monitor(stop_evt, records, interval=0.25):
    while not stop_evt.is_set():
        r = subprocess.run(
            ['nvidia-smi',
             '--query-gpu=index,utilization.gpu,memory.used,memory.total,power.draw',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True)
        ts = time.time()
        for line in r.stdout.strip().splitlines():
            parts = [x.strip() for x in line.split(',')]
            if len(parts) < 5:
                continue
            try:
                records.append(dict(
                    ts=ts, gpu=int(parts[0]),
                    gpu_util=float(parts[1]),
                    mem_mb=float(parts[2]),
                    mem_total=float(parts[3]),
                    power=float(parts[4]) if 'N/A' not in parts[4] else 0.0,
                ))
            except ValueError:
                pass
        time.sleep(interval)

def run_prof(csv_path, B, steps=50, lr=0.001, np_=1):
    prefix = f'mpirun --allow-run-as-root -np {np_} ' if np_ > 1 else ''
    cmd = f'{prefix}./bin/train_vit {csv_path} {steps} {B} {lr}'
    stop = threading.Event()
    recs, cpu_s = [], []

    def _cpu():
        while not stop.is_set():
            cpu_s.append((time.time(), psutil.cpu_percent()))
            time.sleep(0.25)

    gmon = threading.Thread(target=_gpu_monitor, args=(stop, recs), daemon=True)
    cmon = threading.Thread(target=_cpu, daemon=True)
    gmon.start()
    cmon.start()

    t0 = time.time()
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    elapsed = time.time() - t0

    stop.set()
    time.sleep(0.4)

    tput = None
    for line in proc.stdout.splitlines():
        if 'img/s' in line and 'throughput' in line.lower():
            for tok in line.replace('|', ' ').split():
                try:
                    tput = float(tok)
                    break
                except ValueError:
                    pass
            break

    return dict(recs=recs, cpu=cpu_s, elapsed=elapsed,
                tput=tput, stdout=proc.stdout)

CSV = os.environ.get('CSV', '')
assert CSV, 'Run the data cell first (cell 6)'

n_gpus = int(subprocess.check_output(
    'nvidia-smi --query-gpu=name --format=csv,noheader | wc -l',
    shell=True, text=True).strip())
print(f'GPUs detected: {n_gpus}')

prof = {}
BATCH_SIZES = [8, 16, 32, 64]

for B in BATCH_SIZES:
    print(f'  1-GPU  B={B:3d} / 50 steps ...', end=' ', flush=True)
    prof[('1gpu', B)] = run_prof(CSV, B, steps=50, np_=1)
    t = prof[('1gpu', B)]['tput']
    print(f"done  {t} img/s  elapsed={prof[('1gpu', B)]['elapsed']:.1f}s")

if n_gpus >= 2:
    print('  2-GPU  B= 32 / 50 steps (NCCL) ...', end=' ', flush=True)
    prof[('2gpu', 32)] = run_prof(CSV, 32, steps=50, np_=2)
    t = prof[('2gpu', 32)]['tput']
    print(f"done  {t} img/s  elapsed={prof[('2gpu', 32)]['elapsed']:.1f}s")
else:
    print('Single-GPU session - 2-GPU profiling skipped.')

print('Profiling sweep complete.')


## 6b. Plot profiling results

Three figures:

1. **Time-series** — GPU util / mem / power / CPU vs wall-clock time for each B.
2. **Summary bars** — throughput, peak memory, avg GPU util, avg CPU util vs B.
3. **NCCL overhead** — 1-GPU vs 2-GPU GPU utilization at B=32. Visible dips in the 2-GPU curve are the allreduce idle windows (T4s are PCIe-connected, no NVLink).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

BATCH_SIZES = [8, 16, 32, 64]
cmap = plt.cm.tab10(np.linspace(0, 0.8, len(BATCH_SIZES)))

def gpu_ts(recs, gpu_idx=0):
    g = [r for r in recs if r['gpu'] == gpu_idx]
    if not g:
        return [], [], [], []
    t0 = g[0]['ts']
    return (
        [r['ts'] - t0 for r in g],
        [r['gpu_util'] for r in g],
        [r['mem_mb'] / 1024 for r in g],
        [r['power'] for r in g],
    )

# Figure 1: time-series per batch size
fig1, axes = plt.subplots(4, len(BATCH_SIZES), figsize=(5 * len(BATCH_SIZES), 12))
fig1.suptitle('GPU & CPU time-series — 50 steps, 1 GPU', fontsize=13, fontweight='bold')
row_labels = ['GPU util (%)', 'GPU mem (GiB)', 'GPU power (W)', 'CPU util (%)']

for idx, (B, c) in enumerate(zip(BATCH_SIZES, cmap)):
    key = ('1gpu', B)
    if key not in prof:
        for row in range(4):
            axes[row][idx].set_visible(False)
        continue
    exp = prof[key]
    ts, util, mem, pwr = gpu_ts(exp['recs'], 0)

    for row, (data, ylabel) in enumerate([
        (util, 'GPU util %'),
        (mem,  'GPU mem GiB'),
        (pwr,  'GPU power W'),
    ]):
        ax = axes[row][idx]
        ax.plot(ts, data, color=c, linewidth=1.3)
        ax.fill_between(ts, data, alpha=0.12, color=c)
        ax.set_title(f'B={B}', fontsize=10)
        ax.set_xlabel('Time (s)', fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.grid(alpha=0.3)
        ax.set_ylim(bottom=0)

    ax_cpu = axes[3][idx]
    if exp['cpu']:
        ct0 = exp['cpu'][0][0]
        ax_cpu.plot([x[0] - ct0 for x in exp['cpu']], [x[1] for x in exp['cpu']],
                    color=c, linewidth=1.3)
    ax_cpu.set_title(f'B={B}', fontsize=10)
    ax_cpu.set_xlabel('Time (s)', fontsize=8)
    ax_cpu.set_ylabel('CPU util %', fontsize=8)
    ax_cpu.grid(alpha=0.3)
    ax_cpu.set_ylim(0, 105)

for row, label in enumerate(row_labels):
    axes[row][0].set_ylabel(label, fontsize=9)

plt.tight_layout()
plt.savefig('profiling_timeseries.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: profiling_timeseries.png')

# Figure 2: summary bars vs batch size
fig2, axes2 = plt.subplots(2, 2, figsize=(12, 8))
fig2.suptitle('Summary metrics vs batch size — 50 steps, 1 GPU', fontsize=13, fontweight='bold')

tputs    = [prof.get(('1gpu', B), {}).get('tput') or 0 for B in BATCH_SIZES]
peak_mem = [max((r['mem_mb'] for r in prof.get(('1gpu', B), {}).get('recs', [])
                 if r['gpu'] == 0), default=0) / 1024 for B in BATCH_SIZES]
avg_util, avg_cpu = [], []
for B in BATCH_SIZES:
    rs = [r['gpu_util'] for r in prof.get(('1gpu', B), {}).get('recs', []) if r['gpu'] == 0]
    avg_util.append(np.mean(rs) if rs else 0)
    cv = [v for _, v in prof.get(('1gpu', B), {}).get('cpu', [])]
    avg_cpu.append(np.mean(cv) if cv else 0)

labels = [f'B={B}' for B in BATCH_SIZES]
for ax, vals, ylabel, title in [
    (axes2[0, 0], tputs,    'img/s', 'Throughput (img/s)'),
    (axes2[0, 1], peak_mem, 'GiB',   'Peak GPU Memory (GiB)'),
    (axes2[1, 0], avg_util, '%',     'Avg GPU Utilization (%)'),
    (axes2[1, 1], avg_cpu,  '%',     'Avg CPU Utilization (%)'),
]:
    bars = ax.bar(labels, vals, color=cmap, edgecolor='white', linewidth=0.5)
    ax.bar_label(bars, fmt='%.1f', fontsize=9, padding=2)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.grid(alpha=0.3, axis='y')
    ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig('profiling_summary.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: profiling_summary.png')

# Figure 3: NCCL overhead — 1-GPU vs 2-GPU at B=32
if ('2gpu', 32) in prof:
    fig3, (ax_u, ax_m) = plt.subplots(1, 2, figsize=(14, 5))
    fig3.suptitle('NCCL overhead: 1-GPU vs 2-GPU at B=32 (50 steps)',
                  fontsize=13, fontweight='bold')

    ts1,  u1,  m1,  _ = gpu_ts(prof[('1gpu', 32)]['recs'], 0)
    ts2a, u2a, m2a, _ = gpu_ts(prof[('2gpu', 32)]['recs'], 0)
    ts2b, u2b, m2b, _ = gpu_ts(prof[('2gpu', 32)]['recs'], 1)

    for ax, y1, y2a, y2b, ylabel in [
        (ax_u, u1, u2a, u2b, 'GPU util (%)'),
        (ax_m, m1, m2a, m2b, 'GPU mem (GiB)'),
    ]:
        ax.plot(ts1, y1, 'b-', linewidth=1.8, label='1-GPU  GPU0')
        ax.plot(ts2a, y2a, 'r-', linewidth=1.8, label='2-GPU  GPU0')
        ax.plot(ts2b, y2b, 'r--', linewidth=1.3, label='2-GPU  GPU1', alpha=0.8)
        ax.set_xlabel('Time (s)', fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
        ax.set_ylim(bottom=0)

    tput1 = prof[('1gpu', 32)]['tput'] or 0
    tput2 = prof[('2gpu', 32)]['tput'] or 0
    ratio = tput2 / tput1 if tput1 else 0
    ax_u.set_title(
        f'1GPU: {tput1:.0f} img/s  2GPU: {tput2:.0f} img/s  speedup: {ratio:.2f}x'
        ' (dips = NCCL allreduce idle)', fontsize=10)
    ax_m.set_title('GPU memory usage', fontsize=10)

    plt.tight_layout()
    plt.savefig('profiling_nccl.png', dpi=130, bbox_inches='tight')
    plt.show()
    print(f'NCCL: 1GPU={tput1:.0f} img/s  2GPU={tput2:.0f} img/s  speedup={ratio:.2f}x')
else:
    print('No 2-GPU profiling data. Run in a T4x2 session to see NCCL overhead.')

print('All profiling plots saved.')


## 6c. PCIe bandwidth & SM clock — batch-size sweep

Uses **pynvml** (`nvidia-ml-py`) to sample PCIe TX/RX bandwidth (KB/s) and SM clock (MHz) from the driver every 250 ms while training runs in the background.

`nvmlDeviceGetPcieThroughput` averages over 20 ms windows — values reflect
actual bytes flowing on the PCIe bus, not just link width.

- **1-GPU:** only tiny H2D copies (one batch ≈ 100 KB) → PCIe nearly idle between steps
- **2-GPU:** allreduce adds ~`167k params × 4B = 654 KB` per step over PCIe

Also checks for **SM clock throttling** (thermal / power) by comparing current vs max MHz.

⏱ Runtime: same sweep as cell 7 (~2–4 min). Run after cell 7 so `prof` is in scope.

In [ ]:
import subprocess, threading, time, os
import numpy as np

try:
    import pynvml
    pynvml.nvmlInit()
except Exception:
    subprocess.run(['pip', 'install', '-q', 'nvidia-ml-py'], check=True)
    import pynvml
    pynvml.nvmlInit()

def _pcie_monitor(stop_evt, records, interval=0.25):
    n = pynvml.nvmlDeviceGetCount()
    handles = [pynvml.nvmlDeviceGetHandleByIndex(i) for i in range(n)]
    while not stop_evt.is_set():
        ts = time.time()
        for i, h in enumerate(handles):
            try:
                tx  = pynvml.nvmlDeviceGetPcieThroughput(h, pynvml.NVML_PCIE_UTIL_TX_BYTES)
                rx  = pynvml.nvmlDeviceGetPcieThroughput(h, pynvml.NVML_PCIE_UTIL_RX_BYTES)
                sm  = pynvml.nvmlDeviceGetClockInfo(h, pynvml.NVML_CLOCK_SM)
                mem = pynvml.nvmlDeviceGetClockInfo(h, pynvml.NVML_CLOCK_MEM)
                records.append(dict(ts=ts, gpu=i,
                                    tx_kbs=tx, rx_kbs=rx,
                                    sm_mhz=sm, mem_mhz=mem))
            except Exception:
                pass
        time.sleep(interval)

def run_pcie_prof(csv_path, B, steps=50, lr=0.001, np_=1):
    prefix = f'mpirun --allow-run-as-root -np {np_} ' if np_ > 1 else ''
    cmd = f'{prefix}./bin/train_vit {csv_path} {steps} {B} {lr}'
    stop = threading.Event()
    recs = []
    mon = threading.Thread(target=_pcie_monitor, args=(stop, recs), daemon=True)
    mon.start()
    t0 = time.time()
    subprocess.run(cmd, shell=True, capture_output=True)
    elapsed = time.time() - t0
    stop.set()
    time.sleep(0.4)
    return dict(recs=recs, elapsed=elapsed)

CSV = os.environ.get('CSV', '')
assert CSV, 'Run the data cell first (cell 6)'

n_gpus = int(subprocess.check_output(
    'nvidia-smi --query-gpu=name --format=csv,noheader | wc -l',
    shell=True, text=True).strip())

h0 = pynvml.nvmlDeviceGetHandleByIndex(0)
max_sm_mhz  = pynvml.nvmlDeviceGetMaxClockInfo(h0, pynvml.NVML_CLOCK_SM)
max_mem_mhz = pynvml.nvmlDeviceGetMaxClockInfo(h0, pynvml.NVML_CLOCK_MEM)
print(f'T4 max SM clock : {max_sm_mhz} MHz')
print(f'T4 max MEM clock: {max_mem_mhz} MHz')

pcie_prof = {}
BATCH_SIZES = [8, 16, 32, 64]

for B in BATCH_SIZES:
    print(f'  1-GPU  B={B:3d} / 50 steps ...', end=' ', flush=True)
    pcie_prof[('1gpu', B)] = run_pcie_prof(CSV, B, steps=50, np_=1)
    print(f"done  {pcie_prof[('1gpu', B)]['elapsed']:.1f}s")

if n_gpus >= 2:
    print('  2-GPU  B= 32 / 50 steps ...', end=' ', flush=True)
    pcie_prof[('2gpu', 32)] = run_pcie_prof(CSV, 32, steps=50, np_=2)
    print(f"done  {pcie_prof[('2gpu', 32)]['elapsed']:.1f}s")
else:
    print('Single-GPU session - 2-GPU PCIe sweep skipped.')

print('PCIe sweep complete.')


## 6d. PCIe + NCCL overhead plots

Three figures from the PCIe sweep:

1. **Time-series** — PCIe TX/RX (MB/s) and SM clock (MHz) per batch size, 1-GPU.
   TX/RX should be near-zero between steps (only H2D batch copy).    SM clock below max → throttle.
2. **1-GPU vs 2-GPU PCIe comparison at B=32** — allreduce traffic appears as
   sharp periodic spikes on GPU0 and GPU1 simultaneously.
3. **Stacked bar: compute time vs NCCL time per step** — uses elapsed times from
   section 6. Shows what fraction of each step is idle waiting for allreduce.
   Effective NCCL bandwidth is computed as `data / overhead_time`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

BATCH_SIZES = [8, 16, 32, 64]
cmap = plt.cm.tab10(np.linspace(0, 0.8, len(BATCH_SIZES)))

def pcie_ts(recs, gpu_idx=0):
    g = [r for r in recs if r['gpu'] == gpu_idx]
    if not g:
        return [], [], [], []
    t0 = g[0]['ts']
    return (
        [r['ts'] - t0 for r in g],
        [r['tx_kbs'] / 1024 for r in g],   # MB/s
        [r['rx_kbs'] / 1024 for r in g],   # MB/s
        [r['sm_mhz'] for r in g],
    )

# Figure 1: PCIe TX/RX and SM clock per batch size (1-GPU)
fig1, axes = plt.subplots(2, len(BATCH_SIZES), figsize=(5 * len(BATCH_SIZES), 8))
fig1.suptitle('PCIe bandwidth & SM clock — 50 steps, 1 GPU', fontsize=13, fontweight='bold')

for idx, (B, c) in enumerate(zip(BATCH_SIZES, cmap)):
    key = ('1gpu', B)
    if key not in pcie_prof:
        axes[0][idx].set_visible(False)
        axes[1][idx].set_visible(False)
        continue
    ts, tx, rx, sm = pcie_ts(pcie_prof[key]['recs'], 0)

    ax_bw = axes[0][idx]
    ax_bw.plot(ts, tx, color='tab:blue', linewidth=1.2, label='TX (GPU->host)')
    ax_bw.plot(ts, rx, color='tab:orange', linewidth=1.2, label='RX (host->GPU)')
    ax_bw.fill_between(ts, tx, alpha=0.12, color='tab:blue')
    ax_bw.fill_between(ts, rx, alpha=0.12, color='tab:orange')
    ax_bw.set_title(f'B={B}  PCIe (MB/s)', fontsize=10)
    ax_bw.set_xlabel('Time (s)', fontsize=8)
    ax_bw.set_ylabel('MB/s', fontsize=8)
    ax_bw.legend(fontsize=7)
    ax_bw.grid(alpha=0.3)
    ax_bw.set_ylim(bottom=0)

    ax_clk = axes[1][idx]
    ax_clk.plot(ts, sm, color='tab:green', linewidth=1.2, label='SM clock')
    try:
        ax_clk.axhline(max_sm_mhz, color='r', linewidth=0.7, linestyle='--',
                       alpha=0.6, label=f'Max {max_sm_mhz} MHz')
    except NameError:
        pass
    ax_clk.set_title(f'B={B}  SM clock (MHz)', fontsize=10)
    ax_clk.set_xlabel('Time (s)', fontsize=8)
    ax_clk.set_ylabel('MHz', fontsize=8)
    ax_clk.legend(fontsize=7)
    ax_clk.grid(alpha=0.3)
    ax_clk.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('pcie_timeseries.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: pcie_timeseries.png')

# Figure 2: 1-GPU vs 2-GPU PCIe — NCCL spikes visible
if ('2gpu', 32) in pcie_prof and ('1gpu', 32) in pcie_prof:
    fig2, (ax_tx, ax_rx) = plt.subplots(1, 2, figsize=(14, 5))
    fig2.suptitle('PCIe traffic: 1-GPU vs 2-GPU at B=32 — spikes = NCCL allreduce',
                  fontsize=12, fontweight='bold')

    ts1,  tx1,  rx1,  _ = pcie_ts(pcie_prof[('1gpu', 32)]['recs'], 0)
    ts2a, tx2a, rx2a, _ = pcie_ts(pcie_prof[('2gpu', 32)]['recs'], 0)
    ts2b, tx2b, rx2b, _ = pcie_ts(pcie_prof[('2gpu', 32)]['recs'], 1)

    for ax, y1, y2a, y2b, ylabel in [
        (ax_tx, tx1, tx2a, tx2b, 'PCIe TX MB/s  (GPU -> switch/peer)'),
        (ax_rx, rx1, rx2a, rx2b, 'PCIe RX MB/s  (switch/peer -> GPU)'),
    ]:
        ax.plot(ts1, y1, 'b-', linewidth=1.6, label='1-GPU  GPU0', alpha=0.9)
        ax.plot(ts2a, y2a, 'r-', linewidth=1.6, label='2-GPU  GPU0')
        ax.plot(ts2b, y2b, 'r--', linewidth=1.2, label='2-GPU  GPU1', alpha=0.8)
        ax.set_xlabel('Time (s)', fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
        ax.set_ylim(bottom=0)

    plt.tight_layout()
    plt.savefig('pcie_nccl_compare.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('Saved: pcie_nccl_compare.png')
else:
    print('No 2-GPU PCIe data - run in T4x2 session.')

# Figure 3: NCCL overhead stacked bar (uses timing from prof dict, cell 7)
have_timing = 'prof' in dir() and ('1gpu', 32) in prof and ('2gpu', 32) in prof
if have_timing:
    t1_total = prof[('1gpu', 32)]['elapsed']
    t2_total = prof[('2gpu', 32)]['elapsed']
    steps = 50
    nccl_s = max(0.0, t2_total - t1_total)
    ms_compute = t1_total / steps * 1000
    ms_nccl    = nccl_s / steps * 1000

    fig3, (ax_bar, ax_per) = plt.subplots(1, 2, figsize=(11, 5))
    fig3.suptitle('NCCL allreduce overhead vs compute time  (B=32, 50 steps)',
                  fontsize=12, fontweight='bold')

    ax_bar.bar(['1-GPU', '2-GPU'], [t1_total, t1_total],
               color='tab:blue', label='Compute (=1-GPU time)')
    ax_bar.bar(['2-GPU'], [nccl_s], bottom=[t1_total],
               color='tab:orange', label='NCCL overhead')
    ax_bar.set_ylabel('Total time (s)', fontsize=10)
    ax_bar.set_title('Total 50-step time', fontsize=10)
    ax_bar.legend(fontsize=9)
    ax_bar.grid(alpha=0.3, axis='y')

    ax_per.bar(['Compute/step', 'NCCL/step'], [ms_compute, ms_nccl],
               color=['tab:blue', 'tab:orange'])
    ax_per.set_ylabel('ms per step', fontsize=10)
    ax_per.set_title('Per-step breakdown', fontsize=10)
    ax_per.grid(alpha=0.3, axis='y')
    for i, v in enumerate([ms_compute, ms_nccl]):
        ax_per.text(i, v + 0.2, f'{v:.1f} ms', ha='center', fontsize=10)

    params = 167296
    data_kb = params * 4 / 1024
    if nccl_s > 0:
        eff_bw = params * 4 * steps / nccl_s / 1e9
        note = f'{data_kb:.0f} KB/step  |  effective NCCL BW ~ {eff_bw:.2f} GB/s  (PCIe 3.0 x16 peak = 16 GB/s)'
    else:
        note = f'{data_kb:.0f} KB/step  |  no measurable overhead (model too small or timing noise)'
    fig3.text(0.5, 0.01, note, ha='center', fontsize=9, style='italic')

    plt.tight_layout()
    plt.savefig('nccl_overhead.png', dpi=130, bbox_inches='tight')
    plt.show()
    if nccl_s > 0:
        frac = ms_nccl / (ms_compute + ms_nccl) * 100
        print(f'NCCL overhead: {ms_nccl:.1f} ms/step  ({frac:.0f}% of 2-GPU step time)')
        print(f'Effective NCCL BW: {eff_bw:.2f} GB/s  (PCIe 3.0 x16 peak = 16 GB/s)')
    print('Saved: nccl_overhead.png')
else:
    print('Run cell 7 (GPU profiling sweep) with both 1-GPU and 2-GPU to see NCCL breakdown.')

print('PCIe / NCCL plots done.')


## 6e. Roofline analysis

**No training run needed** — purely analytical.

Plots each major operation on the T4 roofline:

- X-axis: **arithmetic intensity** = FLOP / byte (computed from model config)
- Y-axis: **performance ceiling** = min(peak FLOPS, bandwidth × AI)
- Ridge point: 8100 GFLOPS ÷ 300 GB/s = **27 FLOP/byte**

Key insight: with **D=64** this model is almost entirely **memory-bandwidth-bound**.
Even the largest matmuls (QKV proj, FC1, FC2) have AI ≈ 24–26, just below the ridge.
Adam step (AI ≈ 0.3) and embedding (AI ≈ 0.08) are deeply memory-bound.
To become compute-bound you would need D ≥ 256–512.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# T4 hardware limits
peak_gflops = 8100.0   # FP32 peak GFLOPS
peak_bw_gbs = 300.0    # HBM2 bandwidth GB/s
ridge_pt    = peak_gflops / peak_bw_gbs   # = 27 FLOP/byte

# Model config — matches compile-time defaults in train_vit.cu
B, T, D, H, L, C, V = 32, 784, 64, 4, 2, 10, 256
hd = D // H    # head_dim = 16
N  = 167296    # total trainable parameters

# (name, flop, bytes_accessed, group)
ops = [
    ('Embedding',
        B*T*D,
        (B*T*D + B*T*D + B*T*D) * 4,
        'memory'),
    ('LayerNorm',
        5 * B*T*D,
        (B*T*D + 2*D + B*T*D) * 4,
        'memory'),
    ('QKV proj  D->3D',
        2 * B*T * D * (3*D),
        (B*T*D + D*(3*D) + B*T*(3*D)) * 4,
        'matmul'),
    ('Attn QK  [T,hd]x[hd,T]',
        2 * B*H * T * hd * T,
        (B*H*T*hd + B*H*T*hd + B*H*T*T) * 4,
        'attention'),
    ('Attn softmax',
        5 * B*H * T*T,
        2 * B*H*T*T * 4,
        'memory'),
    ('Attn AV  [T,T]x[T,hd]',
        2 * B*H * T*T * hd,
        (B*H*T*T + B*H*T*hd + B*H*T*hd) * 4,
        'attention'),
    ('Attn out proj  D->D',
        2 * B*T * D*D,
        (B*T*D + D*D + B*T*D) * 4,
        'matmul'),
    ('MLP FC1  D->4D',
        2 * B*T * D * (4*D),
        (B*T*D + D*(4*D) + B*T*(4*D)) * 4,
        'matmul'),
    ('GELU',
        8 * B*T * (4*D),
        2 * B*T*(4*D) * 4,
        'memory'),
    ('MLP FC2  4D->D',
        2 * B*T * (4*D) * D,
        (B*T*(4*D) + (4*D)*D + B*T*D) * 4,
        'matmul'),
    ('Mean pool',
        B*T*D,
        (B*T*D + B*D) * 4,
        'memory'),
    ('Adam step',
        8 * N,
        7 * N * 4,
        'optimizer'),
]

color_map = {
    'memory':    '#4878D0',
    'matmul':    '#6ACC65',
    'attention': '#D65F5F',
    'optimizer': '#EE854A',
}

# Annotate only a few representative ops to avoid clutter
labeled = {
    'Embedding', 'Adam step',
    'Attn QK  [T,hd]x[hd,T]', 'QKV proj  D->3D', 'MLP FC1  D->4D',
}

fig, ax = plt.subplots(figsize=(13, 7))

ai_x = np.logspace(-2, 3, 500)
roof = np.minimum(peak_gflops, peak_bw_gbs * ai_x)
ax.loglog(ai_x, roof, 'k-', linewidth=2.2, label='Roofline (T4 FP32)')

ax.axvline(ridge_pt, color='k', linestyle='--', linewidth=0.9, alpha=0.45)
ax.text(ridge_pt * 1.1, 500,
        f'Ridge = {ridge_pt:.0f} FLOP/B', fontsize=8.5)
ax.text(0.013, peak_bw_gbs * 0.014,
        f'{peak_bw_gbs:.0f} GB/s HBM limit', fontsize=8, color='#555', rotation=41)
ax.text(300, peak_gflops * 1.12,
        f'{peak_gflops/1000:.1f} TFLOPS compute limit', fontsize=8, color='#555')

seen = set()
for name, flop, nbytes, grp in ops:
    ai   = flop / nbytes
    ceil = min(peak_gflops, peak_bw_gbs * ai)
    col  = color_map[grp]
    lbl  = grp.capitalize() if grp not in seen else None
    seen.add(grp)
    ax.scatter([ai], [ceil], color=col, s=95, zorder=5,
               edgecolors='white', linewidth=0.8, label=lbl)
    if name in labeled:
        ax.annotate(name, (ai, ceil),
                    textcoords='offset points', xytext=(6, 3),
                    fontsize=8, color=col)

ax.set_xlabel('Arithmetic Intensity  (FLOP / byte)', fontsize=11)
ax.set_ylabel('Performance ceiling  (GFLOPS)', fontsize=11)
ax.set_title(
    f'Roofline  —  T4  |  B={B} T={T} D={D} H={H} L={L}  '
    f'|  ridge = {ridge_pt:.0f} FLOP/byte  '
    '|  left of ridge = memory-bound,  right = compute-bound',
    fontsize=10)
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3, which='both')
ax.set_xlim(0.01, 800)
ax.set_ylim(1, peak_gflops * 3)

plt.tight_layout()
plt.savefig('roofline.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f'{"Operation":<28} {"AI (FLOP/B)":>12}   {"Bound":>10}   {"GFLOP ceiling":>14}')
print('-' * 72)
for name, flop, nbytes, _ in ops:
    ai   = flop / nbytes
    ceil = min(peak_gflops, peak_bw_gbs * ai)
    bound = 'COMPUTE' if ai > ridge_pt else 'MEMORY'
    print(f'{name:<28} {ai:>12.2f}   {bound:>10}   {ceil:>11.0f} GFLOPS')
print(f'\nRidge point: {ridge_pt:.1f} FLOP/byte')
print(f'T4 peak: {peak_gflops/1000:.1f} TFLOPS FP32  |  {peak_bw_gbs:.0f} GB/s HBM2')
print('Note: all ops in this D=64 model sit below or near the ridge -> bottleneck is HBM bandwidth.')
print('Saved: roofline.png')


## 7. Per-step timing breakdown

Reads `training_log.csv` (or the saved 1-/2-GPU copies) and shows how each
step is split across the five pipeline stages timed with CUDA events:

| Column | What it measures |
|--------|------------------|
| `t_h2d_ms`  | Host→device transfer (pixels + labels, pinned memory) |
| `t_fwd_ms`  | Forward pass: embedding → transformer → softmax-CE loss |
| `t_bwd_ms`  | Backward pass: full gradient computation |
| `t_nccl_ms` | NCCL allreduce (gradient sync across GPUs; ≈0 for 1-GPU) |
| `t_adam_ms` | Adam parameter update |

Times are sampled at **every 10th step** (the existing reporting interval).
Plots:
1. **Stacked area over training** — how stage times evolve as training progresses.
2. **Average breakdown bar** — mean time per stage across all logged steps.
3. **1-GPU vs 2-GPU comparison** — shows NCCL cost added in the 2-GPU run.

In [ ]:
import os, pandas as pd, matplotlib.pyplot as plt, numpy as np

TIMING_COLS = ['t_h2d_ms', 't_fwd_ms', 't_bwd_ms', 't_nccl_ms', 't_adam_ms']
LABELS      = ['H2D', 'Forward', 'Backward', 'NCCL', 'Adam']
COLORS      = ['#4878D0', '#6ACC65', '#D65F5F', '#EE854A', '#956CB4']

def load_log(path):
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    if not all(c in df.columns for c in TIMING_COLS):
        print(f'{path}: no timing columns — recompile train_vit.cu and re-run.')
        return None
    return df

logs = {}
for label, path in [('1 GPU', 'log/1gpu_log.csv'), ('2 GPU', 'log/2gpu_log.csv')]:
    df = load_log(path)
    if df is not None:
        logs[label] = df
if not logs:
    df = load_log('log/training_log.csv') or load_log('training_log.csv')
    if df is not None:
        logs['current'] = df

if not logs:
    print('No log files found. Run cells 7–10 first.')
else:
    # ── Figure 1: stacked area over training steps ──────────────────────────
    nplots = len(logs)
    fig1, axes = plt.subplots(1, nplots, figsize=(9 * nplots, 5), squeeze=False)
    fig1.suptitle('Per-step timing — stacked area over training',
                  fontsize=13, fontweight='bold')

    for ax, (run_label, df) in zip(axes[0], logs.items()):
        bottom = np.zeros(len(df))
        for col, lbl, col_c in zip(TIMING_COLS, LABELS, COLORS):
            vals = df[col].values
            ax.fill_between(df['step'], bottom, bottom + vals,
                            label=lbl, color=col_c, alpha=0.82)
            bottom += vals
        ax.set_xlabel('Step', fontsize=10)
        ax.set_ylabel('Time (ms)', fontsize=10)
        ax.set_title(run_label, fontsize=11)
        ax.legend(fontsize=9, loc='upper right')
        ax.grid(alpha=0.3, axis='y')
        ax.set_ylim(bottom=0)

    plt.tight_layout()
    plt.savefig('timing_area.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('Saved: timing_area.png')

    # ── Figure 2: mean breakdown bar ─────────────────────────────────────────
    fig2, axes2 = plt.subplots(1, nplots, figsize=(6 * nplots, 5), squeeze=False)
    fig2.suptitle('Mean time per stage (ms)', fontsize=13, fontweight='bold')

    for ax, (run_label, df) in zip(axes2[0], logs.items()):
        means = [df[c].mean() for c in TIMING_COLS]
        total = sum(means)
        bars = ax.bar(LABELS, means, color=COLORS, edgecolor='white', linewidth=0.6)
        ax.bar_label(bars, fmt='%.2f ms', fontsize=9, padding=3)
        ax.set_ylabel('ms', fontsize=10)
        ax.set_title(f'{run_label}  |  total {total:.1f} ms/step', fontsize=11)
        ax.grid(alpha=0.3, axis='y')
        ax.set_ylim(0, max(means) * 1.25)

        print(f'\n{run_label}  — mean per step ({len(df)} log points):')
        for lbl, m in zip(LABELS, means):
            pct = m / total * 100 if total else 0
            print(f'  {lbl:<10} {m:7.3f} ms  ({pct:5.1f}%)')
        print(f'  {"total":<10} {total:7.3f} ms')

    plt.tight_layout()
    plt.savefig('timing_bars.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('\nSaved: timing_bars.png')

    # ── Figure 3: 1-GPU vs 2-GPU comparison (if both available) ─────────────
    if '1 GPU' in logs and '2 GPU' in logs:
        df1, df2 = logs['1 GPU'], logs['2 GPU']
        means1 = [df1[c].mean() for c in TIMING_COLS]
        means2 = [df2[c].mean() for c in TIMING_COLS]

        x = np.arange(len(LABELS))
        w = 0.35
        fig3, ax3 = plt.subplots(figsize=(10, 5))
        b1 = ax3.bar(x - w/2, means1, w, label='1 GPU', color=COLORS, alpha=0.85,
                     edgecolor='white')
        b2 = ax3.bar(x + w/2, means2, w, label='2 GPU', color=COLORS, alpha=0.55,
                     edgecolor='black', linewidth=0.7, linestyle='--')
        ax3.bar_label(b1, fmt='%.2f', fontsize=8, padding=2)
        ax3.bar_label(b2, fmt='%.2f', fontsize=8, padding=2)
        ax3.set_xticks(x)
        ax3.set_xticklabels(LABELS, fontsize=10)
        ax3.set_ylabel('Mean time (ms)', fontsize=10)
        ax3.set_title(
            '1 GPU vs 2 GPU — mean time per stage\n'
            '(NCCL bar grows from ~0 to allreduce cost; fwd/bwd stay the same)',
            fontsize=11)
        ax3.legend(fontsize=10)
        ax3.grid(alpha=0.3, axis='y')
        plt.tight_layout()
        plt.savefig('timing_1v2gpu.png', dpi=130, bbox_inches='tight')
        plt.show()
        print('Saved: timing_1v2gpu.png')
    else:
        print('Run both 1-GPU and 2-GPU sessions to see the comparison plot.')


## 8. Training runs — long experiments

**Always run these last**, after completing all analysis in sections 1–7.
Each session takes ≈10–15 min.

- **1-GPU:** 8a → 8b, then start a fresh T4×2 session.
- **2-GPU:** 8c → 8d → 8e.

### 8a. Single-GPU run (Adam, 3000 steps)

Optimizer: Adam lr=1e-3, B=32.  
Saves log to `training_log.csv`.  
After this cell finishes — run **8b** to rename the log, then start a **new session** for the 2-GPU run.

In [ ]:
!./bin/train_vit $CSV 3000 32 0.001


### 8b. Save the 1-GPU log

Rename so the 2-GPU run can write a fresh `training_log.csv`.

In [ ]:
import shutil, os
os.makedirs("log", exist_ok=True)
shutil.move("training_log.csv", "log/1gpu_log.csv")
print("Saved as log/1gpu_log.csv  —  size:", os.path.getsize("log/1gpu_log.csv"), "bytes")


### 8c. Two-GPU run (Adam, 3000 steps)

`--allow-run-as-root` is required on Kaggle (kernel runs as root).  
`-np 2` spawns one process per GPU via `cudaSetDevice(rank % devices)`.  
Per-rank batch = 32 → global batch = 64. `adam_step` divides by `B * world` so the update is the mean gradient over the global batch.

In [ ]:
!mpirun --allow-run-as-root -np 2 ./bin/train_vit $CSV 3000 32 0.001


### 8d. Save the 2-GPU log

In [ ]:
import shutil, os
os.makedirs("log", exist_ok=True)
shutil.move("training_log.csv", "log/2gpu_log.csv")
print("Saved as log/2gpu_log.csv  —  size:", os.path.getsize("log/2gpu_log.csv"), "bytes")


### 8e. Loss vs time: 1 GPU vs 2 GPU

X-axis is **wall-clock time** (seconds), not steps — this shows the actual speedup from the second GPU.

In [ ]:
import os, pandas as pd, matplotlib.pyplot as plt, matplotlib.ticker as ticker

logs = {'1 GPU': 'log/1gpu_log.csv', '2 GPU': 'log/2gpu_log.csv'}

fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(14, 5))

for label, path in logs.items():
    if not os.path.exists(path):
        print(f'Файл не найден: {path}')
        continue
    df = pd.read_csv(path)
    smooth = lambda s: s.rolling(window=10, min_periods=1).mean()
    line, = ax_loss.plot(df['elapsed_s'], smooth(df['loss']),
                        label=label, linewidth=2)
    ax_loss.plot(df['elapsed_s'], df['loss'],
                alpha=0.15, color=line.get_color())
    line2, = ax_acc.plot(df['elapsed_s'], smooth(df['accuracy']),
                        label=label, linewidth=2)
    ax_acc.plot(df['elapsed_s'], df['accuracy'],
               alpha=0.15, color=line2.get_color())

ax_loss.set_xlabel('Время, секунды')
ax_loss.set_ylabel('Loss (cross-entropy)')
ax_loss.set_title('Loss vs время')
ax_loss.legend(); ax_loss.grid(alpha=0.3)

ax_acc.set_xlabel('Время, секунды')
ax_acc.set_ylabel('Accuracy')
ax_acc.set_title('Accuracy vs время')
ax_acc.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
ax_acc.legend(); ax_acc.grid(alpha=0.3)

plt.suptitle('1 GPU vs 2 GPU — Adam lr=1e-3, B=32, 3000 steps', fontsize=13)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('График сохранён: training_curves.png')


## Notes

- T4s on Kaggle are PCIe-connected (no NVLink) → NCCL uses PCIe/shared-mem. Speedup on this small model (~167k params) is sub-linear due to allreduce overhead.
- Attention scores scale as `L · B · H · T²`: with `T=784, H=4, L=2, B=32` each of `A_ATTN_PRE` / `A_ATTN` ≈ 0.6 GB — stay well within 15 GB.
- Adam adds one extra float buffer (`d_velocity`, same size as params) vs SGD.
- The log file is flushed after every reporting step (every 10 steps) so you can inspect it while training is running. Each row also contains CUDA-event timings (ms) for the five pipeline stages: H2D, forward, backward, NCCL, Adam.
